# MoPR Semantic Segmentation Inference Pipeline
## Batch Inference, Tile Merging & Output Validation

This notebook runs inference on trained SegFormer model and produces submission-ready outputs:
- Batch inference on test village tiles
- Merge tiles to Cloud-Optimized GeoTIFFs (COG)
- Convert masks to GeoPackage (GPKG) for GIS analysis
- Optional: Compute solar potential from DSM
- Generate village statistics & validation report

## Cell 1: Environment Setup

In [ ]:
!nvidia-smi

In [ ]:
# Install packages
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers timm albumentations
!pip install -q rasterio geopandas rio-cogeo shapely fiona
!pip install -q tqdm numpy pandas opencv-python pillow scikit-image

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

## Cell 2: Mount Drive & Clone Repository

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

repo_path = '/content/mopr-hackathon'
if os.path.exists(repo_path):
    print(f"{repo_path} exists")
else:
    !git clone https://github.com/YOUR_USERNAME/mopr-hackathon.git {repo_path}

os.chdir(repo_path)
print(f"Working directory: {os.getcwd()}")

## Cell 3: Import Libraries & Load Model

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
import json
from tqdm import tqdm
from transformers import SegformerForSemanticSegmentation
import cv2
import rasterio
from rasterio.transform import from_bounds
import geopandas as gpd
from shapely.geometry import shape
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '/content/mopr-hackathon')

from configs.training_config import ModelConfig
from configs.class_weights import CLASS_NAMES, CLASS_COLORS

print("Libraries imported")

In [ ]:
# Load trained model
CHECKPOINT_PATH = "/content/drive/MyDrive/MoPR/checkpoints/best_model.pt"
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Loading model from: {CHECKPOINT_PATH}")
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

# Create model instance
model_cfg = ModelConfig()
model = SegformerForSemanticSegmentation.from_pretrained(
    model_cfg.backbone,
    num_labels=model_cfg.num_classes,
    ignore_mismatched_sizes=True,
)

# Adapt first conv to 4 channels (RGB + DSM)
# Use patch_embeddings[0].proj (NOT patch_embeddings.patch_embeddings.proj)
original_conv = model.segformer.encoder.patch_embeddings[0].proj
original_weights = original_conv.weight.data

new_weight = torch.zeros(
    original_weights.shape[0], 4,
    original_weights.shape[2], original_weights.shape[3],
    device=original_weights.device, dtype=original_weights.dtype,
)
new_weight[:, :3, :, :] = original_weights
new_weight[:, 3:, :, :] = original_weights.mean(dim=1, keepdim=True)

new_conv = nn.Conv2d(
    4, original_weights.shape[0],
    kernel_size=original_conv.kernel_size,
    stride=original_conv.stride,
    padding=original_conv.padding,
    bias=original_conv.bias is not None,
)
new_conv.weight = nn.Parameter(new_weight)
if original_conv.bias is not None:
    new_conv.bias = original_conv.bias

model.segformer.encoder.patch_embeddings[0].proj = new_conv

# Load checkpoint
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(DEVICE)
model.eval()

print(f"Model loaded and moved to {DEVICE}")
print(f"Checkpoint mIoU: {checkpoint.get('mIoU', 'N/A')}")

## Cell 4: Batch Inference Function

In [ ]:
def run_inference_on_tiles(
    tile_dir: str,
    output_dir: str,
    model,
    device,
    batch_size: int = 16,
    num_classes: int = 9,
):
    """
    Run inference on all RGB+DSM tiles in a directory.
    
    Args:
        tile_dir: Directory containing _rgb.tif and _dsm.tif files
        output_dir: Where to save _pred.tif files
        model: Trained model
        device: 'cuda' or 'cpu'
        batch_size: Inference batch size
        num_classes: Number of output classes
    
    Returns:
        List of processed tile filenames
    """
    tile_dir = Path(tile_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Find all RGB tiles
    rgb_files = sorted(tile_dir.glob("*_rgb.tif"))
    print(f"Found {len(rgb_files)} RGB tiles")
    
    processed = []
    
    # Process in batches
    for batch_idx in tqdm(range(0, len(rgb_files), batch_size), desc="Inference batches"):
        batch_files = rgb_files[batch_idx:batch_idx + batch_size]
        batch_images = []
        
        # Load batch
        for rgb_path in batch_files:
            # Read RGB
            rgb = cv2.imread(str(rgb_path))
            rgb = cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB)
            
            # Read DSM
            tile_name = rgb_path.stem.replace('_rgb', '')
            dsm_path = rgb_path.parent / f"{tile_name}_dsm.tif"
            
            if dsm_path.exists():
                dsm = cv2.imread(str(dsm_path), cv2.IMREAD_GRAYSCALE)[:, :, np.newaxis]
                dsm = (dsm.astype(np.float32) - 128.0) / 128.0
            else:
                dsm = np.zeros((rgb.shape[0], rgb.shape[1], 1), dtype=np.float32)
            
            # Stack and normalize RGB
            image = np.concatenate([rgb, dsm], axis=2)
            image = image.astype(np.float32)
            
            # Normalize RGB channels
            mean = np.array([0.485, 0.456, 0.406])
            std = np.array([0.229, 0.224, 0.225])
            image[:, :, :3] = (image[:, :, :3] / 255.0 - mean) / std
            
            batch_images.append(torch.from_numpy(image).permute(2, 0, 1))
        
        # Stack batch
        batch_tensor = torch.stack(batch_images).to(device)
        
        # Inference
        with torch.no_grad():
            outputs = model(pixel_values=batch_tensor)
            logits = outputs.logits
            preds = logits.argmax(dim=1).cpu().numpy().astype(np.uint8)
        
        # Save predictions
        for pred, rgb_path in zip(preds, batch_files):
            tile_name = rgb_path.stem.replace('_rgb', '')
            output_path = output_dir / f"{tile_name}_pred.tif"
            
            # Save as GeoTIFF
            with rasterio.open(
                str(output_path),
                'w',
                driver='GTiff',
                height=pred.shape[0],
                width=pred.shape[1],
                count=1,
                dtype=rasterio.uint8,
            ) as dst:
                dst.write(pred, 1)
            
            processed.append(tile_name)
    
    print(f"\nProcessed {len(processed)} tiles")
    return processed


print("Inference function defined")

## Cell 5: Run Batch Inference

In [ ]:
%%time

# Define paths
TEST_TILES_DIR = "/content/drive/MyDrive/MoPR/data/test_tiles"
INFERENCE_OUTPUT_DIR = "/content/drive/MyDrive/MoPR/inference_masks"

# Run inference
processed_tiles = run_inference_on_tiles(
    tile_dir=TEST_TILES_DIR,
    output_dir=INFERENCE_OUTPUT_DIR,
    model=model,
    device=DEVICE,
    batch_size=16,
    num_classes=9,
)

print(f"\nInference complete. Saved masks to: {INFERENCE_OUTPUT_DIR}")
print(f"Sample processed tiles: {processed_tiles[:5]}")

## Cell 6: Merge Tiles to COG

In [ ]:
import re
import tempfile
from rasterio.merge import merge as rio_merge
from rio_cogeo.cogeo import cog_translate
from rio_cogeo.profiles import cog_profiles


def merge_tiles_to_cog(mask_dir: str, output_dir: str):
    """
    Merge prediction tiles into a single COG per village using rasterio.merge.
    
    Tiles are grouped by village_id parsed from filename patterns like:
      {village_id}_{row}_{col}_pred.tif
    
    Uses rasterio.merge which correctly handles 2D tile grids based on
    geotransform metadata — no manual row/col indexing needed.
    """
    mask_dir = Path(mask_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Group tiles by village
    tile_pattern = re.compile(r"(.+?)_\d+_\d+_pred\.tif$")
    tiles_by_village = {}
    
    for mask_file in sorted(mask_dir.glob("*_pred.tif")):
        match = tile_pattern.match(mask_file.name)
        if match:
            village_name = match.group(1)
        else:
            # Fallback: use everything before last _pred
            village_name = mask_file.stem.replace("_pred", "").rsplit("_", 2)[0]
        
        if village_name not in tiles_by_village:
            tiles_by_village[village_name] = []
        tiles_by_village[village_name].append(mask_file)
    
    print(f"Found {len(tiles_by_village)} villages")
    
    village_names = []
    for village_name, tile_files in tqdm(sorted(tiles_by_village.items()), desc="Merging villages"):
        # Open all tile datasets
        sources = [rasterio.open(str(f)) for f in tile_files]
        
        try:
            # rasterio.merge handles 2D spatial arrangement via geotransforms
            merged_data, merged_transform = rio_merge(sources, method="first")
            reference_crs = sources[0].crs
        finally:
            for src in sources:
                src.close()
        
        # Write to temp file, then COG-translate
        with tempfile.NamedTemporaryFile(suffix=".tif", delete=False) as tmp:
            tmp_path = tmp.name
        
        profile = {
            "driver": "GTiff",
            "dtype": merged_data.dtype,
            "width": merged_data.shape[2],
            "height": merged_data.shape[1],
            "count": merged_data.shape[0],
            "crs": reference_crs,
            "transform": merged_transform,
        }
        
        with rasterio.open(tmp_path, "w", **profile) as dst:
            dst.write(merged_data)
        
        # Convert to COG
        output_path = output_dir / f"{village_name}_segmentation.tif"
        cog_profile = cog_profiles.get("lzw")
        cog_profile.update({"BLOCKSIZE": 512})
        
        cog_translate(tmp_path, str(output_path), dst_kwargs=cog_profile, copy_src_overviews=False)
        Path(tmp_path).unlink(missing_ok=True)
        
        village_names.append(village_name)
        print(f"  {village_name}: {merged_data.shape[2]}x{merged_data.shape[1]} px")
    
    print(f"\nMerged {len(village_names)} villages as COGs")
    return village_names


print("Tile merging function defined (uses rasterio.merge for correct 2D spatial merging)")

In [ ]:
%%time

# Merge tiles
COG_OUTPUT_DIR = "/content/drive/MyDrive/MoPR/segmentation_cogs"
villages = merge_tiles_to_cog(
    mask_dir=INFERENCE_OUTPUT_DIR,
    output_dir=COG_OUTPUT_DIR,
)

print(f"\nVillages processed: {villages}")

## Cell 7: Convert Masks to GeoPackage (GPKG)

In [ ]:
from rasterio.features import shapes as rasterio_shapes
from shapely.geometry import shape as shapely_shape, Polygon

# Class-to-layer mapping (matches configs/class_weights.py)
BUILDING_CLASSES = {1: "RCC", 2: "tile", 3: "tin", 4: "thatched"}
ROAD_CLASSES = {5: "pucca", 6: "kaccha"}
WATER_CLASSES = {7}
VEGETATION_CLASSES = {8}


def mask_to_gpkg(
    mask_path: str,
    output_path: str,
    village_id: str,
    class_names: list,
    simplify_tolerance: float = 0.3,
    min_area_m2: float = 2.0,
):
    """
    Convert raster mask to multi-layer GeoPackage using rasterio.features.shapes.
    
    Uses the mask's native CRS (NOT hardcoded EPSG:4326).
    Creates separate layers: buildings, roads, water_bodies, vegetation.
    """
    with rasterio.open(mask_path) as src:
        mask = src.read(1)
        transform = src.transform
        crs = src.crs
    
    if crs is None:
        raise ValueError(f"Mask {mask_path} has no CRS — cannot produce valid GPKG")
    
    buildings = []
    roads = []
    water = []
    vegetation = []
    
    unique_classes = np.unique(mask)
    
    for class_id in unique_classes:
        if class_id == 0:
            continue
        
        binary = (mask == class_id).astype(np.uint8)
        
        for geom_dict, value in rasterio_shapes(binary, transform=transform):
            if value == 0:
                continue
            
            try:
                geom = shapely_shape(geom_dict)
                if not geom.is_valid:
                    geom = geom.buffer(0)
                if geom.is_empty or geom.area < min_area_m2:
                    continue
                geom = geom.simplify(simplify_tolerance, preserve_topology=True)
                if geom.is_empty:
                    continue
                
                if class_id in BUILDING_CLASSES:
                    buildings.append({
                        "geometry": geom,
                        "roof_type": BUILDING_CLASSES[class_id],
                        "area_m2": round(geom.area, 2),
                        "confidence": 1.0,
                        "village_id": village_id,
                    })
                elif class_id in ROAD_CLASSES:
                    roads.append({
                        "geometry": geom,
                        "road_type": ROAD_CLASSES[class_id],
                        "length_m": round(geom.length, 2),
                        "village_id": village_id,
                    })
                elif class_id in WATER_CLASSES:
                    water.append({
                        "geometry": geom,
                        "area_m2": round(geom.area, 2),
                        "village_id": village_id,
                    })
                elif class_id in VEGETATION_CLASSES:
                    vegetation.append({
                        "geometry": geom,
                        "area_m2": round(geom.area, 2),
                        "village_id": village_id,
                    })
            except Exception:
                continue
    
    # Write layers to GPKG
    output_path = Path(output_path)
    if output_path.exists():
        output_path.unlink()
    
    total = 0
    for layer_name, features in [("buildings", buildings), ("roads", roads), 
                                  ("water_bodies", water), ("vegetation", vegetation)]:
        if features:
            gdf = gpd.GeoDataFrame(features, crs=crs)
            mode = "a" if output_path.exists() else "w"
            gdf.to_file(output_path, layer=layer_name, driver="GPKG", mode=mode)
            total += len(features)
    
    return total


print("GPKG conversion function defined (uses rasterio.features.shapes with native CRS)")

In [ ]:
%%time

# Convert all village masks to GPKG
GPKG_OUTPUT_DIR = "/content/drive/MyDrive/MoPR/segmentation_gpkg"
Path(GPKG_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

for village_name in tqdm(villages, desc="Converting to GPKG"):
    cog_path = Path(COG_OUTPUT_DIR) / f"{village_name}_segmentation.tif"
    gpkg_path = Path(GPKG_OUTPUT_DIR) / f"{village_name}_features.gpkg"
    
    if cog_path.exists():
        try:
            num_features = mask_to_gpkg(
                mask_path=str(cog_path),
                output_path=str(gpkg_path),
                village_id=village_name,
                class_names=CLASS_NAMES,
            )
            print(f"  {village_name}: {num_features} features")
        except Exception as e:
            print(f"  Error processing {village_name}: {e}")

## Cell 8: Generate Village Statistics

In [ ]:
def compute_village_statistics(mask_path: str, class_names: list, pixel_size: float = 0.1):
    """
    Compute area and class distribution statistics for a village.
    
    Args:
        mask_path: Path to segmentation GeoTIFF
        class_names: List of class names
        pixel_size: Size of one pixel in meters (default 0.1m for drone imagery)
    
    Returns:
        Dictionary with statistics
    """
    with rasterio.open(mask_path) as src:
        mask = src.read(1)
    
    stats = {}
    total_pixels = mask.size
    total_area_m2 = total_pixels * (pixel_size ** 2)
    
    for class_id, class_name in enumerate(class_names):
        pixel_count = (mask == class_id).sum()
        area_m2 = pixel_count * (pixel_size ** 2)
        area_ha = area_m2 / 10000.0
        percentage = (pixel_count / total_pixels) * 100.0
        
        stats[class_name] = {
            'pixel_count': int(pixel_count),
            'area_m2': float(area_m2),
            'area_ha': float(area_ha),
            'percentage': float(percentage),
        }
    
    stats['total_area_m2'] = float(total_area_m2)
    stats['total_area_ha'] = float(total_area_m2 / 10000.0)
    
    return stats


# Compute statistics for all villages
all_stats = {}
for village_name in tqdm(villages, desc="Computing statistics"):
    cog_path = Path(COG_OUTPUT_DIR) / f"{village_name}_segmentation.tif"
    if cog_path.exists():
        stats = compute_village_statistics(str(cog_path), CLASS_NAMES)
        all_stats[village_name] = stats

# Create summary table
summary_rows = []
for village_name, stats in all_stats.items():
    row = {'village': village_name}
    for class_name, class_stats in stats.items():
        if class_name != 'total_area_m2' and class_name != 'total_area_ha':
            row[f"{class_name}_ha"] = class_stats['area_ha']
    row['total_ha'] = stats['total_area_ha']
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

print("\n" + "="*100)
print("VILLAGE STATISTICS SUMMARY")
print("="*100)
print(summary_df.to_string(index=False))

# Save statistics
stats_csv = Path(INFERENCE_OUTPUT_DIR) / "village_statistics.csv"
summary_df.to_csv(stats_csv, index=False)
print(f"\nSaved statistics to: {stats_csv}")

## Cell 9: Output Validation

In [ ]:
import fiona

def validate_outputs(cog_dir: str, gpkg_dir: str) -> dict:
    """Validate that all required output files exist and have valid structure."""
    cog_dir = Path(cog_dir)
    gpkg_dir = Path(gpkg_dir)
    
    results = {}
    
    # Check COGs
    cog_files = list(cog_dir.glob("*_segmentation.tif"))
    print(f"\nValidation Report")
    print("=" * 80)
    print(f"\nCOG Files: {len(cog_files)} found")
    
    for cog_file in cog_files:
        try:
            with rasterio.open(str(cog_file)) as src:
                h, w = src.height, src.width
                dtype = src.dtypes[0]
                crs = src.crs
                print(f"  PASS: {cog_file.name}: {w}x{h}, {dtype}, CRS={crs}")
                results[cog_file.stem] = {'cog': 'PASS'}
        except Exception as e:
            print(f"  FAIL: {cog_file.name}: {e}")
            results[cog_file.stem] = {'cog': f'FAIL - {e}'}
    
    # Check GPKGs
    gpkg_files = list(gpkg_dir.glob("*.gpkg"))
    print(f"\nGPKG Files: {len(gpkg_files)} found")
    
    for gpkg_file in gpkg_files:
        try:
            layers = fiona.listlayers(str(gpkg_file))
            issues = []
            
            for layer in layers:
                gdf = gpd.read_file(str(gpkg_file), layer=layer)
                invalid_count = (~gdf.geometry.is_valid).sum()
                if invalid_count > 0:
                    issues.append(f"{layer}: {invalid_count} invalid geoms")
                if gdf.crs is None:
                    issues.append(f"{layer}: missing CRS")
            
            if issues:
                print(f"  WARN: {gpkg_file.name} ({', '.join(layers)})")
                for issue in issues:
                    print(f"    - {issue}")
                results[gpkg_file.stem] = {'gpkg': 'WARN'}
            else:
                total_features = sum(
                    len(gpd.read_file(str(gpkg_file), layer=l)) for l in layers
                )
                print(f"  PASS: {gpkg_file.name}: {total_features} features in {len(layers)} layers")
                results[gpkg_file.stem] = {'gpkg': 'PASS'}
        except Exception as e:
            print(f"  FAIL: {gpkg_file.name}: {e}")
            results[gpkg_file.stem] = {'gpkg': f'FAIL - {e}'}
    
    print(f"\n" + "=" * 80)
    return results


validation = validate_outputs(COG_OUTPUT_DIR, GPKG_OUTPUT_DIR)

## Cell 10: Package Outputs for Submission

In [ ]:
import shutil
import zipfile
import os

SUBMISSION_DIR = "/content/drive/MyDrive/MoPR/submission"
submission_path = Path(SUBMISSION_DIR)
submission_path.mkdir(parents=True, exist_ok=True)

# Create directory structure
output_structure = {
    'segmentation_masks_cog': COG_OUTPUT_DIR,
    'segmentation_vectors_gpkg': GPKG_OUTPUT_DIR,
    'statistics': str(INFERENCE_OUTPUT_DIR / 'village_statistics.csv'),
}

print(f"\nPackaging outputs...")
print(f"Submission directory: {SUBMISSION_DIR}")

# Copy files
for dest_name, src_path in output_structure.items():
    if dest_name == 'statistics':
        # Copy CSV
        dest = submission_path / 'village_statistics.csv'
        shutil.copy2(src_path, dest)
        print(f"✓ Copied statistics CSV")
    else:
        # Copy directory
        dest = submission_path / dest_name
        if dest.exists():
            shutil.rmtree(dest)
        shutil.copytree(src_path, dest)
        print(f"✓ Copied {dest_name} ({len(list(dest.glob('*')))} files)")

# Create README
readme_path = submission_path / "README.md"
with open(readme_path, 'w') as f:
    f.write("""# MoPR Semantic Segmentation Results

## Output Files

### Segmentation Masks (COG format)
- `segmentation_masks_cog/`: Cloud-Optimized GeoTIFFs with class predictions
- Each file: `{village_name}_segmentation.tif`
- 9 classes (0-8): background, RCC_roof, tile_roof, tin_roof, thatched_roof, road_pucca, road_kaccha, water_body, vegetation

### Segmentation Vectors (GeoPackage format)
- `segmentation_vectors_gpkg/`: Vector polygons for each class
- Each file: `{village_name}_segmentation.gpkg`
- GIS-ready format for further analysis

### Statistics
- `village_statistics.csv`: Area statistics per village and class

## Model
- Architecture: SegFormer-B2
- Input: 4-channel (RGB + DSM)
- Training data: Indian village orthophotos

## Usage

To load segmentation masks:
```python
import rasterio
with rasterio.open('village_name_segmentation.tif') as src:
    mask = src.read(1)
```

To load vectors:
```python
import geopandas as gpd
gdf = gpd.read_file('village_name_segmentation.gpkg')
```
""")

print(f"✓ Created README.md")

# Create submission ZIP
zip_path = submission_path.parent / 'MoPR_Semantic_Segmentation_Submission.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(submission_path):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, submission_path.parent)
            zf.write(file_path, arcname)

print(f"✓ Created submission ZIP: {zip_path}")
print(f"  Size: {zip_path.stat().st_size / (1024**2):.2f} MB")

print("\n" + "="*80)
print("SUBMISSION READY")
print("="*80)
print(f"Location: {SUBMISSION_DIR}")
print(f"ZIP file: {zip_path}")